# **V2** Thai Classical Music Generation – Baseline Sequence Modeling




## 0. Prerequisites & Setup

- Reuse normalized symbolic dataset from Stage 2  
- Pitch-only representation (octave stripped)  
- Rest token included as explicit `<REST>` symbol  
- Sequence = flattened token stream per song  

**Goal:**  
Train a baseline conditional LSTM language model on symbolic pitch sequences.

### 0.0 Libs

In [1]:
!pip install mido python-rtmidi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 965.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.6/315.6 kB 5.6 MB/s eta 0:00:00


In [2]:
!wget https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

--2026-02-18 09:12:53--  https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Phonbopit/sarabun-webfont/master/fonts/thsarabunnew-webfont.ttf [following]
--2026-02-18 09:12:53--  https://raw.githubusercontent.com/Phonbopit/sarabun-webfont/master/fonts/thsarabunnew-webfont.ttf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 98308 (96K) [application/octet-stream]
Saving to: ‘thsarabunnew-webfont.ttf’

thsarabunnew-webfon 100%[===================>]  96.00K  --.-KB/s    in 0.01s   

2026-02-18 09:12:53 (6.49 MB/s) - ‘t

In [3]:
!pip install tqdm

In [12]:
!git clone https://github.com/GetomG/Thai-Music-Thesis.git
# If already cloned:
# !git pull


Cloning into 'Thai-Music-Thesis'...
remote: Enumerating objects: 574, done.
remote: Counting objects: 100% (574/574), done.
remote: Compressing objects: 100% (313/313), done.
remote: Total 574 (delta 105), reused 570 (delta 101), pack-reused 0 (from 0)
Receiving objects: 100% (574/574), 13.69 MiB | 28.32 MiB/s, done.
Resolving deltas: 100% (105/105), done.


In [13]:
%cd Thai-Music-Thesis

/content/Thai-Music-Thesis


In [14]:
# ============================================================
# Import Helper Utilities from thai_music_utils
# ============================================================

# 1. Notation Processing
from thai_music_utils.notation_utils import (
    flatten_song_notation,
    normalize_octave_markers,
    notation_to_sequence
)

# 2. Octave Inference (DP-based register guessing)
from thai_music_utils.octave_inference import (
    is_thai_note,
    get_fixed_octave,
    guess_octaves_with_constraints,
    add_octaves_respecting_labels
)

# 3. Preprocessing Utilities
from thai_music_utils.preprocessing import (
    flatten_song_data,
    remove_all_signs
)

# 4. EDA Helpers (Symbolic Analysis)
from thai_music_utils.eda_symbolic_normalization import (
    normalize_token,
    normalize_bar,
    flatten_song,
    THAI_NOTES,
    UP_MARK,
    LOW_MARK,
    REST_TOKEN
)

# 5. EDA Stats
from thai_music_utils.eda_stats import (
    extract_symbols,
    pitch_stats,
    stats_to_df
)

# 6. I/O Utilities
from thai_music_utils.io_utils import (
    save_json_bar_per_line
)

# 7. MIDI Rendering (Ranad-specific)
from thai_music_utils.midi_ranad import (
    generate_ranad_midi
)


In [4]:
#setting Thai fonts

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.font_manager.fontManager.addfont('thsarabunnew-webfont.ttf')
mpl.rc('font', family='TH Sarabun New')
mpl.rcParams["axes.unicode_minus"] = False

In [5]:
import copy

### 0.1 Data Intake

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import json
from pathlib import Path
from collections import defaultdict

BASE = Path("/content/drive/MyDrive/thai_music_data/songs")

songs = []

for motif_dir in BASE.iterdir():
    if not motif_dir.is_dir():
        continue

    motif = motif_dir.name

    for song_dir in motif_dir.iterdir():
        json_dir = song_dir / "json"
        if not json_dir.exists():
            continue

        for json_file in json_dir.glob("*.json"):
            try:
                with open(json_file, "r", encoding="utf-8") as f:
                    data = json.load(f)

                songs.append({
                    "motif": motif,
                    "song": song_dir.name,
                    "path": str(json_file),
                    "data": data
                })

            except Exception as e:
                print(f"⚠️ Skipped {json_file}: {e}")

In [8]:
print(f"Total songs loaded: {len(songs)}")

by_motif = defaultdict(int)
for s in songs:
    by_motif[s["motif"]] += 1

for motif, n in by_motif.items():
    print(f"{motif}: {n} songs")

Total songs loaded: 44
ลาว: 18 songs
พม่า: 5 songs
ไทยเดิม: 2 songs
จีน: 4 songs
แขก: 8 songs
เขมร: 7 songs


In [11]:
# from collections import defaultdict

# note_count_by_motif = defaultdict(int)
# song_count_by_motif = defaultdict(int)

# for s in songs:
#     motif = s["motif"]
#     note_count_by_motif[motif] += len(s["pitch_sequence"])
#     song_count_by_motif[motif] += 1

# print("=== Note Count per Motif ===\n")

# for motif in sorted(note_count_by_motif.keys()):
#     total_notes = note_count_by_motif[motif]
#     total_songs = song_count_by_motif[motif]
#     avg_len = total_notes / total_songs

#     print(f"{motif}:")
#     print(f"  Total notes (including rests): {total_notes}")
#     print(f"  Songs: {total_songs}")
#     print(f"  Avg notes per song: {avg_len:.2f}")
#     print()

=== Note Count per Motif ===

จีน:
  Total notes (including rests): 1740
  Songs: 4
  Avg notes per song: 435.00

พม่า:
  Total notes (including rests): 971
  Songs: 5
  Avg notes per song: 194.20

ลาว:
  Total notes (including rests): 5706
  Songs: 17
  Avg notes per song: 335.65

เขมร:
  Total notes (including rests): 5936
  Songs: 7
  Avg notes per song: 848.00

แขก:
  Total notes (including rests): 5725
  Songs: 8
  Avg notes per song: 715.62

ไทยเดิม:
  Total notes (including rests): 836
  Songs: 2
  Avg notes per song: 418.00



#### **Filter only Khmer**

In [ ]:
# songs = [s for s in songs if s["motif"] == "เขมร"]

# print("Total Khmer songs:", len(songs))

Total Khmer songs: 7


### Motif Filter

In [15]:
### Motif FILTER

### *0.2* Symbolic Normalization & Sequence Construction

#### 1️⃣ Normalize + Flatten

Reuse the earlier symbolic normalization logic, but simplify it for sequence modeling.

**Goal:**
- Clean and standardize symbolic tokens
- Decide whether to keep or remove register (pitch-only vs pitch+register)
- Convert structured JSON (section → bar → token) into one linear sequence per song
- Prepare clean token streams ready for vocabulary building and LSTM training

Output:
Each song → one flat symbolic sequence (list of tokens)

#### Main Helpers

In [9]:
THAI_NOTES = set("ดรมฟซลท")
UP_MARK = "ํ"
LOW_MARK = "ฺ"

def normalize_token(token):
    """
    Convert token into pitch tokens + compressed rest tokens.

    Rest compression rule:
    - Any number of consecutive dashes is decomposed into
      chunks of <REST_4>, <REST_3>, <REST_2>, <REST_1>
    """

    if not isinstance(token, str):
        return ["<REST_1>"]

    token = token.strip()

    # ---- Pure rest token ----
    if set(token) == {"-"}:
        dash_count = len(token)
        rests = []

        while dash_count > 0:
            if dash_count >= 4:
                rests.append("<REST_4>")
                dash_count -= 4
            elif dash_count >= 3:
                rests.append("<REST_3>")
                dash_count -= 3
            elif dash_count >= 2:
                rests.append("<REST_2>")
                dash_count -= 2
            else:
                rests.append("<REST_1>")
                dash_count -= 1

        return rests

    # ---- Mixed pitch token ----
    out = []
    i = 0

    while i < len(token):
        ch = token[i]

        if ch == "-":
            # count consecutive dashes
            dash_count = 0
            while i < len(token) and token[i] == "-":
                dash_count += 1
                i += 1

            while dash_count > 0:
                if dash_count >= 4:
                    out.append("<REST_4>")
                    dash_count -= 4
                elif dash_count >= 3:
                    out.append("<REST_3>")
                    dash_count -= 3
                elif dash_count >= 2:
                    out.append("<REST_2>")
                    dash_count -= 2
                else:
                    out.append("<REST_1>")
                    dash_count -= 1

        elif ch in THAI_NOTES:
            out.append(ch)
            i += 1

            # skip octave mark
            if i < len(token) and token[i] in {UP_MARK, LOW_MARK}:
                i += 1

        else:
            i += 1

    return out if out else ["<REST_1>"]

def song_to_pitch_sequence(song_json):
    """
    Convert full song JSON into one flat pitch sequence.
    Handles:
    - normal string tokens
    - dict blocks like {"นำ": [...]}, {"ตาม": [...]}
    - nested lists
    """

    sequence = []

    def process_token(tok):
        # Case 1: string token
        if isinstance(tok, str):
            sequence.extend(normalize_token(tok))

        # Case 2: dict (นำ / ตาม block)
        elif isinstance(tok, dict):
            for key in tok:
                for inner_tok in tok[key]:
                    process_token(inner_tok)

        # Case 3: list (nested structure)
        elif isinstance(tok, list):
            for inner_tok in tok:
                process_token(inner_tok)

    for section in song_json.get("sections", []):
        for bar in section.get("bars", []):
            for tok in bar:
                process_token(tok)

    return sequence

#### Apply to songs

In [ ]:
print(songs[2]["song"])

ลาวคำหอม


In [10]:
#Apply to songs
for s in songs:
    s["pitch_sequence"] = song_to_pitch_sequence(s["data"])

print(len(songs[2]["pitch_sequence"]))
print(songs[2]["pitch_sequence"][:80])

511
['<REST_3>', 'ม', '<REST_1>', 'ซ', 'ซ', 'ซ', '<REST_1>', 'ม', '<REST_1>', 'ล', '<REST_1>', 'ซ', 'ซ', 'ซ', '<REST_1>', 'ล', '<REST_1>', 'ซ', '<REST_1>', 'ม', '<REST_1>', 'ซ', '<REST_2>', 'ล', 'ด', 'ซ', 'ล', '<REST_1>', 'ด', '<REST_4>', '<REST_3>', 'ด', '<REST_1>', 'ด', 'ด', 'ด', '<REST_1>', 'ด', '<REST_1>', 'ด', '<REST_4>', '<REST_3>', 'ซ', '<REST_2>', 'ด', 'ล', 'ซ', 'ม', '<REST_1>', 'ซ', '<REST_4>', '<REST_3>', 'ด', '<REST_4>', '<REST_1>', 'ร', '<REST_1>', 'ม', '<REST_3>', 'ซ', '<REST_3>', 'ร', '<REST_1>', 'ม', '<REST_1>', 'ร', '<REST_1>', 'ด', 'ด', 'ด', '<REST_3>', 'ร', 'ม', 'ร', 'ด', 'ล', 'ซ', 'ม', 'ซ', 'ล']


In [11]:
from collections import defaultdict

note_count_by_motif = defaultdict(int)
song_count_by_motif = defaultdict(int)

for s in songs:
    motif = s["motif"]
    note_count_by_motif[motif] += len(s["pitch_sequence"])
    song_count_by_motif[motif] += 1

print("=== Note Count per Motif ===\n")

for motif in sorted(note_count_by_motif.keys()):
    total_notes = note_count_by_motif[motif]
    total_songs = song_count_by_motif[motif]
    avg_len = total_notes / total_songs

    print(f"{motif}:")
    print(f"  Total notes (including rests): {total_notes}")
    print(f"  Songs: {total_songs}")
    print(f"  Avg notes per song: {avg_len:.2f}")
    print()

=== Note Count per Motif ===

จีน:
  Total notes (including rests): 1740
  Songs: 4
  Avg notes per song: 435.00

พม่า:
  Total notes (including rests): 971
  Songs: 5
  Avg notes per song: 194.20

ลาว:
  Total notes (including rests): 6005
  Songs: 18
  Avg notes per song: 333.61

เขมร:
  Total notes (including rests): 5936
  Songs: 7
  Avg notes per song: 848.00

แขก:
  Total notes (including rests): 5725
  Songs: 8
  Avg notes per song: 715.62

ไทยเดิม:
  Total notes (including rests): 836
  Songs: 2
  Avg notes per song: 418.00



#### 2️⃣ Build Vocabulary (Token → Integer Mapping)

Neural networks cannot process symbolic tokens directly.  
We must convert each pitch token into a numeric ID.

Steps:
- Collect all unique tokens from all songs
- Assign each token a unique integer
- Build two mappings:
  - `token_to_id`
  - `id_to_token`

This defines:
- The model vocabulary
- The input/output dimension for the LSTM

In [18]:
from collections import Counter

# Collect all tokens across songs
all_tokens = []

for s in songs:
    all_tokens.extend(s["pitch_sequence"])

# Unique vocabulary
vocab = sorted(set(all_tokens))

# Create mappings
token_to_id = {tok: i for i, tok in enumerate(vocab)}
id_to_token = {i: tok for tok, i in token_to_id.items()}

vocab_size = len(vocab)

print("Vocabulary:", vocab)
print("Vocab size:", vocab_size)

Vocabulary: ['<REST_1>', '<REST_2>', '<REST_3>', '<REST_4>', 'ซ', 'ด', 'ท', 'ฟ', 'ม', 'ร', 'ล']
Vocab size: 11


In [ ]:
# for s in songs:
#     for tok in s["pitch_sequence"]:
#         if tok in ["<", "R", "E", "S", "T", ">"]:
#             print("BROKEN:", s["song"])
#             break

#### 3️⃣ Convert Songs to Integer Sequences

Neural networks cannot process symbolic tokens directly.  
We must convert each pitch token into a numeric ID.

Steps:
- Take each song’s `pitch_sequence`
- Replace each token with its corresponding integer ID using `token_to_id`
- Store the new numeric sequence (e.g., `id_sequence`) inside each song

This produces:
- One integer sequence per song  
- Shape per song: `(sequence_length,)`
- Vocabulary size = `vocab_size`

These integer sequences will be used to:
- Create training samples (input → next token prediction)
- Feed into the LSTM model
- Compute loss over predicted next-token probabilities

In [19]:
# 3️⃣ Convert Songs to Integer Sequences

for s in songs:
    s["id_sequence"] = [
        token_to_id[token]
        for token in s["pitch_sequence"]
        if token in token_to_id
    ]

# sanity check
print("Example song length:", len(songs[2]["id_sequence"]))
print("First 30 token IDs:", songs[2]["id_sequence"][:30])

Example song length: 511
First 30 token IDs: [2, 8, 0, 4, 4, 4, 0, 8, 0, 10, 0, 4, 4, 4, 0, 10, 0, 4, 0, 8, 0, 4, 1, 10, 5, 4, 10, 0, 5, 3]


#### 4️⃣ Prepare Training Sequences (LSTM Input Construction)

The LSTM does not see full songs at once.

Instead, we train it using sliding windows:

Given a sequence:
ด ร ม ซ ด ร ด ...

We create training samples like:

```
Input (length = seq_len) → Target
[ด ร ม ซ] → ด
[ร ม ซ ด] → ร
[ม ซ ด ร] → ด
```

This teaches the model:
"Given the previous N notes, predict the next note."

Steps:
- Choose a sequence length (e.g., 16)
- Slide window across every song
- Convert token IDs into (X, y) pairs
- X shape: (num_samples, seq_len)
- y shape: (num_samples,)

In [ ]:
for s in songs:
    s["id_sequence"] = [
        token_to_id[token]
        for token in s["pitch_sequence"]
    ]

In [ ]:
import numpy as np

SEQ_LEN = 16  # number of previous notes to condition on

X = []
y = []

for s in songs:
    ids = s["id_sequence"]

    if len(ids) <= SEQ_LEN:
        continue

    for i in range(len(ids) - SEQ_LEN):
        X.append(ids[i:i+SEQ_LEN])
        y.append(ids[i+SEQ_LEN])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5824, 16)
y shape: (5824,)


## 1️⃣ LSTM Model Definition

We now define a baseline LSTM language model.

Goal:
- Input: sequence of 16 token IDs
- Output: probability distribution over next token

Architecture:
- Embedding layer (token → dense vector)
- LSTM layer
- Linear output layer
- Softmax handled by CrossEntropyLoss

This is a standard neural language model setup.

In [16]:
# ============================================================
# 0. PREREQUISITES
# ============================================================

import numpy as np
import random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [20]:
import torch
import torch.nn as nn

# ============================================================
# LSTM Language Model (Improved Version)
# - 2-layer stacked LSTM
# - Dropout between layers
# - Predict next token from last hidden state
# ============================================================

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2, dropout=0.25):
        super().__init__()

        # 1️⃣ Embedding Layer
        # Converts token IDs → dense vectors
        # Shape: (batch, seq_len) → (batch, seq_len, embed_dim)
        # --------------------------------------------------------
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim
        )

        # 2️⃣ Stacked LSTM
        # num_layers=2 → hierarchical pattern modeling
        # dropout applied BETWEEN LSTM layers
        # --------------------------------------------------------
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )

        # 3️⃣ Final Linear Layer
        # Maps final hidden state → vocabulary logits
        # Shape: (batch, hidden_dim) → (batch, vocab_size)
        # --------------------------------------------------------
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        """
        x shape: (batch_size, seq_len)
        """

        # Step 1: Convert tokens to embeddings
        x = self.embedding(x)

        # Step 2: Pass through stacked LSTM
        # output shape: (batch, seq_len, hidden_dim)
        output, _ = self.lstm(x)

        # Step 3: Take last timestep only
        last_hidden = output[:, -1, :]

        # Step 4: Project to vocab size
        logits = self.fc(last_hidden)

        return logits


# Instantiate model
# ------------------------------------------------------------
model = LSTMLanguageModel(vocab_size).to(device)

print(model)

LSTMLanguageModel(
  (embedding): Embedding(11, 64)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.25)
  (fc): Linear(in_features=128, out_features=11, bias=True)
)


### 1️⃣ Training Setup

Now we train the LSTM as a next-token predictor.

Task:
Given 16 previous tokens → predict the next token.

We define:

• Loss: CrossEntropyLoss  
  - Standard for multi-class classification  
  - Compares predicted distribution vs true next-token  

• Optimizer: Adam  
  - Stable for sequence models  
  - Good default for LSTM  

• Training loop:
  1. Forward pass
  2. Compute loss
  3. Backpropagation
  4. Update weights
  5. Repeat for multiple epochs

Goal:
Minimize next-token prediction loss.

In [ ]:
from tqdm import tqdm

In [ ]:
from tqdm import tqdm

# Convert to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.long).to(device)
y_tensor = torch.tensor(y, dtype=torch.long).to(device)

# Training config
epochs = 20
batch_size = 64
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

num_samples = X_tensor.shape[0]


In [ ]:

for epoch in range(epochs):

    model.train()
    total_loss = 0.0

    progress_bar = tqdm(
        range(0, num_samples, batch_size),
        desc=f"Epoch {epoch+1}/{epochs}"
    )

    for i in progress_bar:

        X_batch = X_tensor[i:i+batch_size]
        y_batch = y_tensor[i:i+batch_size]

        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / (num_samples // batch_size)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

Epoch 1/20: 100%|██████████| 91/91 [00:06<00:00, 14.05it/s, loss=1.94]


Epoch 1 Average Loss: 2.1136


Epoch 2/20: 100%|██████████| 91/91 [00:05<00:00, 16.36it/s, loss=1.69]


Epoch 2 Average Loss: 1.8490


Epoch 3/20: 100%|██████████| 91/91 [00:04<00:00, 18.54it/s, loss=1.56]


...

Epoch 18 Average Loss: 0.6587


Epoch 19/20: 100%|██████████| 91/91 [00:05<00:00, 18.13it/s, loss=0.757]


Epoch 19 Average Loss: 0.6135


Epoch 20/20: 100%|██████████| 91/91 [00:06<00:00, 15.00it/s, loss=0.697]

Epoch 20 Average Loss: 0.5788


In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/thai_music_data")
WEIGHTS_DIR = BASE / "weights"

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

print("Weights directory:", WEIGHTS_DIR)

Weights directory: /content/drive/MyDrive/thai_music_data/weights


In [ ]:
import torch

torch.save(model.state_dict(), WEIGHTS_DIR / "lstm_pitch_only_khmer_1.pth")
print("Model saved.")

Model saved.


## 2️⃣ Generation (Sampling)

Now we use the trained LSTM to generate new symbolic pitch sequences.

The model works as a next-token predictor:
- Input: 16-token context window
- Output: probability distribution over next token

Generation procedure:
1. Start with a seed sequence
2. Predict next token
3. Append prediction
4. Slide window forward
5. Repeat

We use **temperature sampling**:
- Low temperature (<1.0) → safer, repetitive
- High temperature (>1.0) → more creative, unstable

This lets us observe whether the model has learned meaningful Thai melodic structure.

In [21]:
model = LSTMLanguageModel(vocab_size).to(device)
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/thai_music_data/weights/lstm_pitch_only_khmer_1.pth",
        map_location=device
    )
)
model.eval()

LSTMLanguageModel(
  (embedding): Embedding(11, 64)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.25)
  (fc): Linear(in_features=128, out_features=11, bias=True)
)

In [22]:
import torch
import torch.nn.functional as F

def generate_sequence(
    model,
    seed_ids,
    max_new_tokens=100,
    temperature=1.0,
    seed=None   # ← add this
):

    model.eval()


    # 🔴 ADD THIS
    if seed is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    generated = seed_ids.copy()

    for _ in range(max_new_tokens):

        context = generated[-16:]
        x = torch.tensor(context, dtype=torch.long).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x)

        logits = logits / temperature
        probs = F.softmax(logits, dim=-1)

        next_id = torch.multinomial(probs, num_samples=1).item()
        generated.append(next_id)

    return generated

def decode_ids(id_sequence):
  return [id_to_token[i] for i in id_sequence]

### Generate from song

In [ ]:
def generate_from_song(
    song_idx,
    model,
    songs,
    seq_len=16,
    max_new_tokens=120,
    temperature=0.8
):
    """
    Generate continuation from a selected song
    and print side-by-side comparison.
    """

    model.eval()

    song = songs[song_idx]
    song_name = song["song"]

    print(f"\n=== Song Index: {song_idx} | Song: {song_name} ===")

    # ---- Seed ----
    seed_ids = song["id_sequence"][:seq_len]

    # ---- Generate ----
    generated_ids = generate_sequence(
        model,
        seed_ids=seed_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature
    )

    # ---- Decode ----
    original_ids = song["id_sequence"][:seq_len + max_new_tokens]

    original_tokens = decode_ids(original_ids)
    generated_tokens = decode_ids(generated_ids)

    # ---- Print ----
    print("\nSEED:")
    print(original_tokens[:seq_len])

    print("\nORIGINAL continuation:")
    print(original_tokens[seq_len:seq_len + 60])

    print("\nGENERATED continuation:")
    print(generated_tokens[seq_len:seq_len + 60])

In [ ]:
generate_from_song(
    song_idx=22,
    model=model,
    songs=songs,
    seq_len=16,
    max_new_tokens=120,
    temperature=1.2
)


=== Song Index: 22 | Song: แขกขาว ===

SEED:
['<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', 'ร', '<REST>', 'ร', 'ร', 'ร', '<REST>', 'ร', '<REST>', 'ร']

ORIGINAL continuation:
['<REST>', 'ม', '<REST>', 'ด', '<REST>', 'ร', '<REST>', 'ม', '<REST>', 'ฟ', '<REST>', 'ซ', '<REST>', '<REST>', '<REST>', 'ร', '<REST>', '<REST>', '<REST>', 'ซ', 'ล', 'ท', 'ด', 'ร', '<REST>', 'ร', 'ร', 'ร', '<REST>', 'ร', '<REST>', 'ร', '<REST>', 'ด', '<REST>', 'ด', '<REST>', '<REST>', '<REST>', '<REST>', 'ล', '<REST>', '<REST>', 'ด', 'ล', 'ซ', 'ฟ', '<REST>', 'ซ', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>']

GENERATED continuation:
['ด', 'ร', 'ฟ', 'ซ', 'ฟ', 'ร', 'ฟ', 'ด', 'ร', 'ร', '<REST>', 'ร', 'ร', 'ร', '<REST>', 'ร', '<REST>', 'ด', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', '<REST>', 'ล', '<REST>', 'ซ', 'ฟ', 'ซ', '<REST>', 'ล', '<REST>', '<REST>', 'ซ', 'ล', 'ด', 'ร', 'ฟ', 'ซ', 'ฟ', 'ร

### Generate from raw frags

In [23]:
# Sort songs by length of pitch_sequence (descending)
sorted_songs = sorted(
    songs,
    key=lambda s: len(s["pitch_sequence"]),
    reverse=True
)

print("Top 5 Longest Songs:\n")

for i, s in enumerate(sorted_songs[:5]):
    print(f"{i+1}. {s['song']} | Motif: {s['motif']} | Length: {len(s['pitch_sequence'])}")

Top 5 Longest Songs:

1. เขมรโพธิสัตว์ | Motif: เขมร | Length: 1679
2. ลาวเสี่ยงเทียน (เถา) | Motif: ลาว | Length: 1492
3. แขกสาย | Motif: แขก | Length: 1344
4. เขมรพวง | Motif: เขมร | Length: 1007
5. แขกขาว | Motif: แขก | Length: 859


### **Helpers**

In [24]:
def generate_from_fragment(
    fragment_tokens,
    model,
    token_to_id,
    id_to_token,
    seq_len=16,
    max_new_tokens=120,
    temperature=0.8,
    bar_size=32,  # 1 bar = 32 slots
    seed=None   # ← add this
):
    """
    Generate continuation from manually provided JSON-style fragment.

    Display:
    - <REST> shown as "-"
    - New line every bar (32 tokens)
    """

    model.eval()

    # ---- Normalize fragment ----
    normalized = []
    for tok in fragment_tokens:
        normalized.extend(normalize_token(tok))

    fragment_ids = [
        token_to_id[t]
        for t in normalized
        if t in token_to_id
    ]

    if not fragment_ids:
        print("⚠️ Fragment produced no valid tokens.")
        return None

    # ---- Left pad ----
    if len(fragment_ids) < seq_len:
        pad_len = seq_len - len(fragment_ids)
        fragment_ids = [token_to_id["<REST>"]] * pad_len + fragment_ids
    else:
        fragment_ids = fragment_ids[-seq_len:]

    # ---- Generate ----
    generated_ids = generate_sequence(
        model,
        seed_ids=fragment_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        seed=seed
    )

    generated_tokens = [id_to_token[i] for i in generated_ids]


    # ---- Convert any REST token to "-" (for display only) ----
    pretty_tokens = [
        "-" if t.startswith("<REST") else t
        for t in generated_tokens[seq_len:]
]
    print("\nGENERATED continuation:\n")

    # ---- Print per bar ----
    for i in range(0, len(pretty_tokens), bar_size):
        bar = pretty_tokens[i:i+bar_size]

        # group every 4 tokens (1 slot)
        grouped = [
            "".join(bar[j:j+4])
            for j in range(0, len(bar), 4)
        ]

        print(", ".join(grouped))

    return generated_tokens

In [30]:
def combine_fragment_and_generated(
    fragment_tokens,
    generated_tokens,
    normalize_token,
    seq_len=16
):
    """
    Combine original fragment (preserving octave marks)
    with generated continuation.

    Returns dash-based slot list.
    """

    # -----------------------------
    # 1️⃣ Keep fragment EXACTLY as is
    # -----------------------------
    fragment_slots = fragment_tokens.copy()

    # -----------------------------
    # 2️⃣ Remove seed duplication from generated
    # -----------------------------
    continuation = generated_tokens[seq_len:]

    # -----------------------------
    # 3️⃣ Convert generated tokens back to dash string
    # -----------------------------
    generated_parts = []

    for tok in continuation:
        if tok.startswith("<REST_"):
            k = int(tok.replace("<REST_", "").replace(">", ""))
            generated_parts.append("-" * k)
        else:
            generated_parts.append(tok)

    generated_string = "".join(generated_parts)

    # -----------------------------
    # 4️⃣ Split generated part into 4-char slots
    # -----------------------------
    generated_slots = [
        generated_string[i:i+4]
        for i in range(0, len(generated_string), 4)
    ]

    # -----------------------------
    # 5️⃣ Combine fragment + generated
    # -----------------------------
    combined_slots = fragment_slots + generated_slots

    return combined_slots

In [34]:
def slots_to_song_data(slots, title="Generated"):

    bars = [
        slots[i:i+8]
        for i in range(0, len(slots), 8)
    ]

    return {
        "title": title,
        "sections": [
            {
                "name": "Generated",
                "bars": bars
            }
        ]
    }

In [35]:
THAI_NOTES = "ดรมฟซลท"

thai_base = {'ด': 58, 'ร': 60, 'ม': 62, 'ฟ': 63, 'ซ': 65, 'ล': 67, 'ท': 69}
octave_offset = {1: -12, 2: 0, 3: 12}

# Only these note–octave combos are allowed (16 total)
allowed_oct = {
    'ด': [2, 3],      # ด, ดํ
    'ร': [2, 3],      # ร, รํ
    'ม': [2, 3],      # ม, มํ
    'ฟ': [2, 3],      # ฟ, ฟํ
    'ซ': [1, 2, 3],   # ซฺ ซ ซํ
    'ล': [1, 2, 3],   # ลฺ ล ลํ
    'ท': [1, 2],      # ทฺ ท   (no ทํ)
}


## Frag Demo

In [29]:
fragment = [ "-ดํดํดํ", "-ล-ดํ", "-ฟํ-รํ", "-ดํ-ล", "-รํดํล", "ดํลซฟ", "ดรมฟ", "มฟซล", "-ดํดํดํ", "-รํ-ฟํ", "-ซํฟํรํ", "-ดํ-ล", "-ลดํซ", "ลซฟร", "ฟรดร", "ฟซ-ฟ" ]


generated_tokens = generate_from_fragment(
    fragment_tokens=fragment,
    model=model,
    token_to_id=token_to_id,
    id_to_token=id_to_token,
    seq_len=16,
    max_new_tokens=120,
    temperature=0.8,
    seed=34

)


# GENERATED continuation:

# รด-ร, ---ด, ดดฟร, ดล-ซ, ฟ-ซ-, ฟร-ฟ, รด-ร, ---ฟ
# -ฟ-ฟ, ซ-ฟซ, ล--ซ, -ซ-ฟ, -ฟฟฟ, -ฟ-ฟ, -ลดร, -ฟ-ซ
# ลซดล, -ซ-ฟ, -ดรฟ, -ร-ด, -ล-ซ, -ฟฟฟ, ฟซฟร, -ฟ-ล
# ดร-ซ, ดลดด, ดดรด, ลดรฟ, -ร-ฟ, -ร-ฟ


GENERATED continuation:

ซ-ฟซ, ล-ซฟ, รฟซฟ, ร--ด, -ร-ด, ดดฟร, ดล-ซ, ฟ-ซ-
ฟ-ซซ, ซซ-ฟ, --ฟ-, ฟฟฟฟ, ฟ-ฟ-, ฟ-ฟ-, ร-ด-, ฟ-รด
รฟด-, รดลด, -ร-ฟ, ---ฟ, ซล-ด, -ร-ร, ร-รร, รร-ฟ
ร-ซฟ, ลซ-ล, ดลซฟ, ร-ทล, ทรทล, ซ-ซล


In [31]:
combined_slots = combine_fragment_and_generated(
    fragment_tokens=fragment,
    generated_tokens=generated_tokens,
    normalize_token=normalize_token
)

print(combined_slots[:80])

['-ดํดํดํ', '-ล-ดํ', '-ฟํ-รํ', '-ดํ-ล', '-รํดํล', 'ดํลซฟ', 'ดรมฟ', 'มฟซล', '-ดํดํดํ', '-รํ-ฟํ', '-ซํฟํรํ', '-ดํ-ล', '-ลดํซ', 'ลซฟร', 'ฟรดร', 'ฟซ-ฟ', 'ซ-ฟซ', 'ล-ซฟ', 'รฟซฟ', 'ร---', '--ด-', 'ร-ดด', 'ดฟรด', 'ล--ซ', 'ฟ-ซ-', 'ฟ--ซ', 'ซซซ-', 'ฟ---', '----', 'ฟ-ฟฟ', 'ฟฟฟ-', 'ฟ-ฟ-', 'ฟ-ร-', 'ด-ฟ-', '--รด', 'รฟด-', '---ร', 'ดลด-', 'ร-ฟ-', '----', '----', 'ฟซล-', 'ด-ร-', '-รร-', '-รรร', 'ร--ฟ', 'ร--ซ', 'ฟลซ-', '-ลดล', 'ซฟร-', 'ทลทร', 'ทลซ-', '-ซล']


In [39]:
song_data_generated = slots_to_song_data(combined_slots)
import pprint

pprint.pprint(song_data_generated, width=120)

{'sections': [{'bars': [['-ดํดํดํ', '-ล-ดํ', '-ฟํ-รํ', '-ดํ-ล', '-รํดํล', 'ดํลซฟ', 'ดรมฟ', 'มฟซล'],
                        ['-ดํดํดํ', '-รํ-ฟํ', '-ซํฟํรํ', '-ดํ-ล', '-ลดํซ', 'ลซฟร', 'ฟรดร', 'ฟซ-ฟ'],
                        ['ซ-ฟซ', 'ล-ซฟ', 'รฟซฟ', 'ร---', '--ด-', 'ร-ดด', 'ดฟรด', 'ล--ซ'],
                        ['ฟ-ซ-', 'ฟ--ซ', 'ซซซ-', 'ฟ---', '----', 'ฟ-ฟฟ', 'ฟฟฟ-', 'ฟ-ฟ-'],
                        ['ฟ-ร-', 'ด-ฟ-', '--รด', 'รฟด-', '---ร', 'ดลด-', 'ร-ฟ-', '----'],
                        ['----', 'ฟซล-', 'ด-ร-', '-รร-', '-รรร', 'ร--ฟ', 'ร--ซ', 'ฟลซ-'],
                        ['-ลดล', 'ซฟร-', 'ทลทร', 'ทลซ-', '-ซล']],
               'name': 'Generated'}],
 'title': 'Generated'}


In [40]:
## Add octave via automated octave logic
song_data_auto = add_octaves_respecting_labels(song_data_generated)

In [41]:
from pprint import pprint

pprint(song_data_auto, width=120, sort_dicts=False)

{'title': 'Generated',
 'sections': [{'name': 'Generated',
               'bars': [['-ดํดํดํ', '-ล-ดํ', '-ฟํ-รํ', '-ดํ-ล', '-รํดํล', 'ดํลซฟ', 'ดรมฟ', 'มฟซล'],
                        ['-ดํดํดํ', '-รํ-ฟํ', '-ซํฟํรํ', '-ดํ-ล', '-ลดํซํ', 'ลํซํฟํรํ', 'ฟํรํดํรํ', 'ฟํซํ-ฟํ'],
                        ['ซํ-ฟํซํ', 'ลํ-ซํฟํ', 'รํฟํซํฟํ', 'รํ---', '--ดํ-', 'รํ-ดํดํ', 'ดํฟํรํดํ', 'ล--ซ'],
                        ['ฟ-ซ-', 'ฟ--ซ', 'ซซซ-', 'ฟ---', '----', 'ฟ-ฟฟ', 'ฟฟฟ-', 'ฟ-ฟ-'],
                        ['ฟ-ร-', 'ด-ฟ-', '--รด', 'รฟด-', '---ร', 'ดลฺด-', 'ร-ฟ-', '----'],
                        ['----', 'ฟซล-', 'ดํ-รํ-', '-รํรํ-', '-รํรํรํ', 'รํ--ฟํ', 'รํ--ซ', 'ฟลซ-'],
                        ['-ลดํล', 'ซฟร-', 'ทฺลฺทฺร', 'ทฺลฺซฺ-', '-ซฺลฺ']]}]}


In [42]:
combined_slots = []

for sec in song_data_auto["sections"]:
    for bar in sec["bars"]:
        if isinstance(bar, list):
            combined_slots.extend(bar)

In [48]:
# export_slots_to_midi(
#     combined_slots,
#     save_path="/content/generated_khmer_4.mid"
# )
sequence_string = "".join(combined_slots)

generate_ranad_midi(
    combined_slots,
    output_path="/content/generated_khmer_4.mid",
    bpm=150,
)

TypeError: expected string or bytes-like object, got 'list'

## Export to MIDI stuff

In [33]:
import re
import random
from mido import Message, MidiFile, MidiTrack, MetaMessage, bpm2tempo

def export_slots_to_midi(
    combined_slots,
    save_path="/content/generated_output.mid",
    bpm=150,
    global_transpose=12,
    play_in_octave_pairs=True
):

    # --------------------------------
    # 1) Merge slots into sequence
    # --------------------------------
    sequence = "".join(combined_slots)

    # --------------------------------
    # 2) Handle octave marks
    # --------------------------------
    LOW_DOT = "\u0E3A"

    def convert_low_notes(seq):
        return re.sub(rf"([ดรมฟซลท]){LOW_DOT}+", lambda m: m.group(1) + "1", seq)

    def convert_high_notes(seq):
        return re.sub(r"([ดรมฟซลท])ํ", lambda m: m.group(1) + "3", seq)

    sequence = convert_low_notes(sequence)
    sequence = convert_high_notes(sequence)

    # --------------------------------
    # 3) MIDI setup
    # --------------------------------
    thai_base = {'ด': 58, 'ร': 60, 'ม': 62, 'ฟ': 63, 'ซ': 65, 'ล': 67, 'ท': 69}
    octave_offset = {1: -12, 2: 0, 3: +12}

    midi = MidiFile(ticks_per_beat=480)
    track = MidiTrack()
    midi.tracks.append(track)

    track.append(MetaMessage('set_tempo', tempo=bpm2tempo(bpm), time=0))
    track.append(MetaMessage('time_signature', numerator=4, denominator=4, time=0))
    track.append(Message('program_change', program=12, time=0))

    ticks_per_slot = 240
    breath_gap = ticks_per_slot // 8
    note_pattern = re.compile(r'([ดรมฟซลท])(\d)?')

    cursor = 0
    last_end = 0

    # --------------------------------
    # 4) Main sequencer
    # --------------------------------
    for match in re.finditer(note_pattern, sequence):
        start_idx = match.start()
        symbol, octave_tag = match.groups()

        # Count dashes before note
        dashes_between = sequence[last_end:start_idx].count('-')
        cursor += dashes_between * ticks_per_slot

        base = thai_base[symbol]
        octave = int(octave_tag) if octave_tag else 2
        main_pitch = base + octave_offset[octave] + global_transpose

        pitches = [main_pitch]
        if play_in_octave_pairs:
            pair = main_pitch - 12
            pitches = sorted({main_pitch, pair})

        # Count dashes after note
        j = match.end()
        dash_count = 0
        while j < len(sequence) and sequence[j] == '-':
            dash_count += 1
            j += 1

        # Roll logic
        if dash_count >= 2:
            roll_duration = ticks_per_slot
            roll_step = ticks_per_slot // 4
            elapsed = 0
            toggle = 0

            while elapsed < roll_duration:
                p = pitches[toggle % len(pitches)]
                vel = random.choice([76, 80, 84])
                t = cursor if elapsed == 0 else roll_step
                track.append(Message("note_on", note=p, velocity=vel, time=t))
                track.append(Message("note_off", note=p, velocity=vel, time=roll_step))
                cursor = 0
                elapsed += roll_step
                toggle += 1

            cursor += (dash_count - 1) * ticks_per_slot + breath_gap
            last_end = j
            continue

        # Normal note
        track.append(Message("note_on", note=pitches[0], velocity=80, time=cursor))
        for p in pitches[1:]:
            track.append(Message("note_on", note=p, velocity=80, time=0))

        track.append(Message("note_off", note=pitches[0], velocity=80, time=ticks_per_slot))
        for p in pitches[1:]:
            track.append(Message("note_off", note=p, velocity=80, time=0))

        cursor = 0
        cursor += dash_count * ticks_per_slot
        last_end = j

    cursor += sequence[last_end:].count('-') * ticks_per_slot
    track.append(MetaMessage("end_of_track", time=cursor))

    midi.save(save_path)
    print("✅ MIDI saved:", save_path)

## Octave guessing helpers

In [36]:
def guess_octaves_with_constraints(notes, fixed_octaves,
                                   prefer_octave=2,
                                   low_pitch=58-12,
                                   high_pitch=69+12,
                                   max_jump=4        # 👈 ADD THIS new
                                    ):

    """
    notes: list of Thai note symbols, e.g. ['ด','ร','ม',...]
    fixed_octaves: list of same length, each in {1,2,3} or None
        - if fixed_octaves[i] is not None, that note MUST be in that octave.
    returns: list of octave tags [1,2,3,...]
    """
    N = len(notes)
    if N == 0:
        return []

    INF = 10**9

    # dp[i][oct] uses oct in {1,2,3}, but we only fill allowed_oct[note]
    dp = [[INF]*4 for _ in range(N)]
    prev = [[None]*4 for _ in range(N)]

    def pitch(note, octv):
        return thai_base[note] + octave_offset[octv]

    def range_penalty(p):
        if p < low_pitch or p > high_pitch:
            return 10 + 0.5 * min(abs(p-low_pitch), abs(p-high_pitch))
        return 0.0

    # ----- init for first note -----
    n0 = notes[0]
    fixed0 = fixed_octaves[0]
    allowed0 = allowed_oct.get(n0, [prefer_octave])

    for o in allowed0:
        if fixed0 is not None and o != fixed0:
            continue  # not allowed by constraint
        p0 = pitch(n0, o)
        cost = 0.0
        cost += abs(o - prefer_octave) * 1.0
        cost += range_penalty(p0)
        dp[0][o] = cost
        prev[0][o] = None

    # ----- transitions -----
    for i in range(1, N):
        n_prev = notes[i-1]
        n_cur  = notes[i]
        fixed_i = fixed_octaves[i]

        allowed_cur = allowed_oct.get(n_cur, [prefer_octave])

        for o_cur in allowed_cur:
            if fixed_i is not None and o_cur != fixed_i:
                continue  # this octave is not allowed at i

            p_cur = pitch(n_cur, o_cur)
            rp = range_penalty(p_cur)

            best_cost = INF
            best_prev_o = None

            # previous can only be octaves that were ever reachable
            for o_prev in [1, 2, 3]:
                if dp[i-1][o_prev] >= INF:
                    continue  # impossible path

                p_prev = pitch(n_prev, o_prev)
                base_cost = dp[i-1][o_prev]

                interval = abs(p_cur - p_prev)

                # smoothness cost
                if interval <= 4:
                    jump_cost = interval * 0.5
                elif interval <= 7:
                    jump_cost = 2 + (interval-4)*1.0
                else:
                    jump_cost = 10 + (interval-7)*2.0

                octave_switch = 0.0 if o_cur == o_prev else 1.5

                total = base_cost + jump_cost + octave_switch + rp

                if total < best_cost:
                    best_cost = total
                    best_prev_o = o_prev

            dp[i][o_cur] = best_cost
            prev[i][o_cur] = best_prev_o

    # ----- backtrack -----
    # choose best last octave among allowed octaves for last note
    last_note = notes[-1]
    allowed_last = allowed_oct.get(last_note, [prefer_octave])

    best_last = None
    best_val = INF
    for o in allowed_last:
        if dp[N-1][o] < best_val:
            best_val = dp[N-1][o]
            best_last = o

    if best_last is None:
        best_last = prefer_octave  # fallback (shouldn’t happen)

    tags = [0]*N
    tags[N-1] = best_last

    for i in range(N-1, 0, -1):
        tags[i-1] = prev[i][tags[i]]

    # Enforce fixed octaves explicitly (in case of numerical tie weirdness)
    for i, fo in enumerate(fixed_octaves):
        if fo is not None:
            tags[i] = fo

    return tags

In [37]:
import copy

LOW_DOT = "ฺ"
HIGH_DOT = "ํ"

def is_thai_note(ch):
    return ch in THAI_NOTES

def get_fixed_octave(token, i):
    """
    token: string like 'มํรด' ; i is index of Thai note char.
    Look at the char after i; if it encodes octave, return 1/2/3, else None.
    """
    if i + 1 >= len(token):
        return None
    nxt = token[i+1]
    if nxt == LOW_DOT:
        return 1
    if nxt == HIGH_DOT:
        return 3
    if nxt.isdigit() and nxt in "123":
        return int(nxt)
    return None


def add_octaves_respecting_labels(song_data):
    """
    - Reads song_data with some notes already tagged (ฺ, ํ, or digits).
    - Uses DP to guess octaves for the remaining untagged notes.
    - Returns a NEW song_data with ฺ/ํ inserted for those untagged notes.
    """
    data = copy.deepcopy(song_data)

    notes = []          # all Thai note symbols in time order
    fixed_octaves = []  # same length, 1/2/3 or None

    # ---------- 1) Collect notes + constraints ----------
    for sec in data["sections"]:
        for bar in sec["bars"]:
            if isinstance(bar, list):
                token_lists = [bar]
            elif isinstance(bar, dict):
                token_lists = [bar.get("นำ", []), bar.get("ตาม", [])]
            else:
                continue

            for tokens in token_lists:
                for token in tokens:
                    i = 0
                    while i < len(token):
                        ch = token[i]
                        if is_thai_note(ch):
                            fo = get_fixed_octave(token, i)
                            notes.append(ch)
                            fixed_octaves.append(fo)
                        i += 1

    # nothing to do
    if not notes:
        return data

    # ---------- 2) Guess octaves with constraints ----------
    tags = guess_octaves_with_constraints(notes, fixed_octaves)
    idx = 0  # index in notes/tags

    # ---------- 3) Inject ฺ / ํ for untagged notes ----------
    for sec in data["sections"]:
        for bi, bar in enumerate(sec["bars"]):
            if isinstance(bar, list):
                token_lists = [bar]
                dict_mode = False
            elif isinstance(bar, dict):
                token_lists = [bar.get("นำ", []), bar.get("ตาม", [])]
                dict_mode = True
            else:
                continue

            for tl_i, tokens in enumerate(token_lists):
                new_tokens = []
                for token in tokens:
                    new_chars = []
                    i = 0
                    while i < len(token):
                        ch = token[i]
                        new_chars.append(ch)

                        if is_thai_note(ch):
                            fo = get_fixed_octave(token, i)
                            tag_oct = tags[idx]
                            idx += 1

                            # if already labeled -> keep as-is
                            if fo is not None:
                                pass
                            else:
                                # add accent based on tag_oct
                                if tag_oct == 1:
                                    new_chars.append(LOW_DOT)
                                elif tag_oct == 3:
                                    new_chars.append(HIGH_DOT)
                                # 2 = mid → nothing extra
                        i += 1

                    new_tokens.append("".join(new_chars))

                # write back
                if not dict_mode:
                    sec["bars"][bi] = new_tokens
                else:
                    if tl_i == 0:
                        bar["นำ"] = new_tokens
                    else:
                        bar["ตาม"] = new_tokens

    return data

In [38]:
def flatten_song_data(nested_song_data):
    flat_data = {
        "title": nested_song_data.get("title", "Untitled"),
        "sections": []
    }
    for top_level_sec in nested_song_data.get("sections", []):
        if "bars" in top_level_sec:
            # Already a flat section
            flat_data["sections"].append(top_level_sec)
        elif "sections" in top_level_sec:
            # Nested sections, flatten them
            for sub_sec in top_level_sec["sections"]:
                # Combine names for clarity
                new_sec_name = f"{top_level_sec['name']} {sub_sec.get('name', '')}".strip()
                flat_data["sections"].append({
                    "name": new_sec_name,
                    "bars": sub_sec.get("bars", [])
                })
        # else: ignore sections without 'bars' or nested 'sections'
    return flat_data